In [4]:
import duckdb
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import xgboost as xgb
import mlflow
import warnings
warnings.filterwarnings('ignore')

# Research DB — never production
con = duckdb.connect('/media/vjl2dev/b1eb2f9b-513e-4494-a9fa-9c137dd6f81b/media/vjerome2/Extreme Pro/kairos_phase4/data/kairos_research.duckdb', read_only=True)

# Verify connection and table availability
tables = con.execute("SHOW TABLES").fetchdf()
print("=== Tables in research DB ===")
print(tables.to_string())

# Verify date range
print("\n=== Date range in feat_targets ===")
print(con.execute("""
    SELECT 
        MIN(date) as min_date,
        MAX(date) as max_date,
        COUNT(DISTINCT date) as trading_days,
        COUNT(DISTINCT ticker) as tickers,
        COUNT(*) as total_rows
    FROM feat_targets
""").fetchdf().to_string())

=== Tables in research DB ===
                name
0          feat_beta
1   feat_fundamental
2   feat_momentum_v2
3  feat_price_action
4       feat_targets
5    feat_vol_sizing
6  sep_base_academic
7            tickers

=== Date range in feat_targets ===
    min_date   max_date  trading_days  tickers  total_rows
0 2015-01-02 2025-12-31          2762     2335     5288553


In [5]:
FEATURES = [
    'earnings_yield', 'fcf_yield', 'roa', 'book_to_market',
    'operating_margin', 'roe',
    'vol_21', 'vol_63', 'vol_blend',
    'beta_21d', 'beta_63d', 'beta_252d', 'resid_vol_63d',
    'hl_ratio', 'range_pct', 'ret_21d', 'ret_5d',
    'mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'mom_12_1', 'reversal_1m',
]

df = con.execute("""
    SELECT
        t.ticker,
        t.date,
        t.ret_5d_f,
        t.label_5d_up,
        f.earnings_yield, f.fcf_yield, f.roa, f.book_to_market,
        f.operating_margin, f.roe,
        v.vol_21, v.vol_63, v.vol_blend,
        b.beta_21d, b.beta_63d, b.beta_252d, b.resid_vol_63d,
        p.hl_ratio, p.range_pct, p.ret_21d, p.ret_5d,
        m.mom_1m, m.mom_3m, m.mom_6m, m.mom_12m, m.mom_12_1, m.reversal_1m
    FROM feat_targets t
    LEFT JOIN feat_fundamental f ON t.ticker = f.ticker AND t.date = f.date
    LEFT JOIN feat_vol_sizing v  ON t.ticker = v.ticker AND t.date = v.date
    LEFT JOIN feat_beta b        ON t.ticker = b.ticker AND t.date = b.date
    LEFT JOIN feat_price_action p ON t.ticker = p.ticker AND t.date = p.date
    LEFT JOIN feat_momentum_v2 m ON t.ticker = m.ticker AND t.date = m.date
    WHERE t.ret_5d_f IS NOT NULL
      AND t.date >= '2015-01-01'
    ORDER BY t.date, t.ticker
""").fetchdf()

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

print(f"=== Feature matrix loaded ===")
print(f"Shape:   {df.shape}")
print(f"Years:   {df['year'].min()} - {df['year'].max()}")
print(f"Tickers: {df['ticker'].nunique()}")
print(f"\n=== Missing value rates (features only) ===")
missing = df[FEATURES].isnull().mean().sort_values(ascending=False)
print(missing[missing > 0].to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Feature matrix loaded ===
Shape:   (5288553, 28)
Years:   2015 - 2025
Tickers: 2335

=== Missing value rates (features only) ===
operating_margin    0.050170
mom_12m             0.049427
mom_6m              0.048838
mom_3m              0.048041
mom_1m              0.047543
reversal_1m         0.047543
mom_12_1            0.047334
beta_252d           0.040497
earnings_yield      0.035716
resid_vol_63d       0.020061
fcf_yield           0.015215
book_to_market      0.011309
roe                 0.010534
roa                 0.010482
vol_21              0.010120
vol_blend           0.010120
vol_63              0.010120
beta_21d            0.010120
beta_63d            0.010120
range_pct           0.003353
hl_ratio            0.003353
ret_21d             0.003353
ret_5d              0.003353


In [6]:
# Join sector labels using the confirmed clean join
sectors = con.execute("""
    SELECT DISTINCT ticker, sector
    FROM tickers
    WHERE sector IS NOT NULL
      AND ticker != 'N/A'
""").fetchdf()

df = df.merge(sectors, on='ticker', how='left')

# Compute equal-weight sector mean return per date
# Exclude the stock itself to avoid lookahead
sector_mean = (
    df.groupby(['date', 'sector'])['ret_5d_f']
    .mean()
    .rename('sector_ret_5d')
    .reset_index()
)

df = df.merge(sector_mean, on=['date', 'sector'], how='left')

# Sector-neutral target
df['ret_5d_sector_neutral'] = df['ret_5d_f'] - df['sector_ret_5d']

# Validation checks
print("=== Sector join coverage ===")
print(f"Rows with sector label:    {df['sector'].notna().sum():,}")
print(f"Rows missing sector label: {df['sector'].isna().sum():,}")
print(f"Unique sectors:            {df['sector'].nunique()}")

print("\n=== Sector-neutral target sanity check ===")
print(f"ret_5d_f               mean: {df['ret_5d_f'].mean():.6f}  std: {df['ret_5d_f'].std():.6f}")
print(f"ret_5d_sector_neutral  mean: {df['ret_5d_sector_neutral'].mean():.6f}  std: {df['ret_5d_sector_neutral'].std():.6f}")

print("\n=== Rows with missing sector-neutral target ===")
print(f"{df['ret_5d_sector_neutral'].isna().sum():,}")

print("\n=== Sector distribution ===")
print(df.groupby('sector')['ticker'].nunique().sort_values(ascending=False).to_string())

=== Sector join coverage ===
Rows with sector label:    5,288,553
Rows missing sector label: 0
Unique sectors:            11

=== Sector-neutral target sanity check ===
ret_5d_f               mean: 0.004026  std: 0.511722
ret_5d_sector_neutral  mean: 0.000000  std: 0.508200

=== Rows with missing sector-neutral target ===
0

=== Sector distribution ===
sector
Healthcare                508
Technology                364
Industrials               318
Financial Services        298
Consumer Cyclical         278
Basic Materials           119
Energy                    118
Consumer Defensive        108
Real Estate               103
Communication Services     75
Utilities                  46


In [ ]:
TEST_YEARS = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

XGB_PARAMS_CLF = {
    'n_estimators': 500,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'min_child_weight': 50,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'early_stopping_rounds': 50,
    'device': 'cuda',
    'random_state': 42,
    'verbosity': 0,
}

def get_splits_old(df, test_year):
    """Current broken regime — val is last 10% of training data."""
    train_all = df[df['year'] < test_year].copy()
    test      = df[df['year'] == test_year].copy()

    n_val = int(len(train_all) * 0.10)
    train = train_all.iloc[:-n_val]
    val   = train_all.iloc[-n_val:]

    return train, val, test

def get_splits_new(df, test_year):
    """Fixed regime — val is a dedicated full year before test year."""
    train = df[df['year'] < test_year - 1].copy()
    val   = df[df['year'] == test_year - 1].copy()
    test  = df[df['year'] == test_year].copy()

    return train, val, test

# Inspect what each regime produces for a single test year (2023)
# to make the difference concrete before running full CV
test_year = 2023

tr_old, va_old, te_old = get_splits_old(df, test_year)
tr_new, va_new, te_new = get_splits_new(df, test_year)

print("=== Split sizes for test_year=2023 ===")
print(f"{'':30s} {'OLD':>12s}  {'NEW':>12s}")
print(f"{'Train rows':30s} {len(tr_old):>12,}  {len(tr_new):>12,}")
print(f"{'Val rows':30s} {len(va_old):>12,}  {len(va_new):>12,}")
print(f"{'Test rows':30s} {len(te_old):>12,}  {len(te_new):>12,}")

print(f"\n=== Validation date range ===")
print(f"OLD val: {va_old['date'].min().date()} → {va_old['date'].max().date()}")
print(f"NEW val: {va_new['date'].min().date()} → {va_new['date'].max().date()}")

print(f"\n=== OLD val covers what market period? ===")
print(f"Years in old val: {va_old['year'].value_counts().sort_index().to_string()}")

In [ ]:
def run_cv(df, get_splits_fn, regime_name):
    results = []

    for test_year in TEST_YEARS:
        train, val, test = get_splits_fn(df, test_year)

        # Skip if any split is empty
        if len(train) == 0 or len(val) == 0 or len(test) == 0:
            print(f"  {test_year}: SKIPPED — empty split")
            continue

        X_train = train[FEATURES].values
        y_train = train['label_5d_up'].values
        X_val   = val[FEATURES].values
        y_val   = val['label_5d_up'].values
        X_test  = test[FEATURES].values
        y_test_ret = test['ret_5d_sector_neutral'].values

        model = xgb.XGBClassifier(**XGB_PARAMS_CLF)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        n_trees = model.best_iteration + 1
        y_pred  = model.predict_proba(X_test)[:, 1]
        ic, _   = spearmanr(y_test_ret, y_pred)

        results.append({
            'regime':    regime_name,
            'test_year': test_year,
            'n_trees':   n_trees,
            'ic':        ic,
            'n_train':   len(train),
            'n_val':     len(val),
            'n_test':    len(test),
        })

        print(f"  {test_year}: trees={n_trees:>4d}  IC={ic:+.4f}")

    return pd.DataFrame(results)

print("=" * 55)
print("REGIME: OLD (last 10% of train as val)")
print("=" * 55)
results_old = run_cv(df, get_splits_old, 'old')

print("\n" + "=" * 55)
print("REGIME: NEW (dedicated validation year)")
print("=" * 55)
results_new = run_cv(df, get_splits_new, 'new')

In [ ]:
def summarize(results_df, label):
    ic = results_df['ic']
    trees = results_df['n_trees']
    print(f"\n=== {label} ===")
    print(f"{'':25s} {'IC':>10s}  {'Trees':>10s}")
    print(f"{'Mean':25s} {ic.mean():>10.4f}  {trees.mean():>10.1f}")
    print(f"{'Std':25s} {ic.std():>10.4f}  {trees.std():>10.1f}")
    print(f"{'Min':25s} {ic.min():>10.4f}  {trees.min():>10.0f}")
    print(f"{'Max':25s} {ic.max():>10.4f}  {trees.max():>10.0f}")
    print(f"{'IC Sharpe (mean/std)':25s} {ic.mean()/ic.std():>10.3f}")
    print(f"{'% folds IC > 0':25s} {(ic > 0).mean():>10.0%}")
    print(f"\nPer-fold detail:")
    print(results_df[['test_year','n_trees','ic']].to_string(index=False))

summarize(results_old, "OLD regime (last 10% of train as val)")
summarize(results_new, "NEW regime (dedicated validation year)")

print("\n=== Tree count comparison per fold ===")
compare = results_old[['test_year','n_trees','ic']].copy()
compare.columns = ['test_year','trees_old','ic_old']
compare['trees_new'] = results_new['n_trees'].values
compare['ic_new']    = results_new['ic'].values
compare['trees_diff'] = compare['trees_new'] - compare['trees_old']
compare['ic_diff']    = compare['ic_new'] - compare['ic_old']
print(compare.to_string(index=False))

In [ ]:
def run_cv_dual_ic(df, get_splits_fn, regime_name):
    results = []

    for test_year in TEST_YEARS:
        train, val, test = get_splits_fn(df, test_year)

        if len(train) == 0 or len(val) == 0 or len(test) == 0:
            print(f"  {test_year}: SKIPPED — empty split")
            continue

        X_train = train[FEATURES].values
        y_train = train['label_5d_up'].values
        X_val   = val[FEATURES].values
        y_val   = val['label_5d_up'].values
        X_test  = test[FEATURES].values

        y_test_raw     = test['ret_5d_f'].values
        y_test_neutral = test['ret_5d_sector_neutral'].values

        model = xgb.XGBClassifier(**XGB_PARAMS_CLF)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        n_trees        = model.best_iteration + 1
        y_pred         = model.predict_proba(X_test)[:, 1]
        ic_raw, _      = spearmanr(y_test_raw, y_pred)
        ic_neutral, _  = spearmanr(y_test_neutral, y_pred)

        results.append({
            'regime':      regime_name,
            'test_year':   test_year,
            'n_trees':     n_trees,
            'ic_raw':      ic_raw,
            'ic_neutral':  ic_neutral,
        })

        print(f"  {test_year}: trees={n_trees:>4d}  IC_raw={ic_raw:+.4f}  IC_neutral={ic_neutral:+.4f}")

    return pd.DataFrame(results)

print("=" * 65)
print("REGIME: OLD (last 10% of train as val)")
print("=" * 65)
results_old2 = run_cv_dual_ic(df, get_splits_old, 'old')

print("\n" + "=" * 65)
print("REGIME: NEW (dedicated validation year)")
print("=" * 65)
results_new2 = run_cv_dual_ic(df, get_splits_new, 'new')

print("\n=== SUMMARY — IC vs raw ret_5d_f ===")
print(f"{'':30s} {'OLD':>10s}  {'NEW':>10s}")
print(f"{'Mean IC_raw':30s} {results_old2['ic_raw'].mean():>10.4f}  {results_new2['ic_raw'].mean():>10.4f}")
print(f"{'Std IC_raw':30s} {results_old2['ic_raw'].std():>10.4f}  {results_new2['ic_raw'].std():>10.4f}")
print(f"{'IC Sharpe (raw)':30s} {results_old2['ic_raw'].mean()/results_old2['ic_raw'].std():>10.3f}  {results_new2['ic_raw'].mean()/results_new2['ic_raw'].std():>10.3f}")
print(f"{'Mean IC_neutral':30s} {results_old2['ic_neutral'].mean():>10.4f}  {results_new2['ic_neutral'].mean():>10.4f}")
print(f"{'Std IC_neutral':30s} {results_old2['ic_neutral'].std():>10.4f}  {results_new2['ic_neutral'].std():>10.4f}")
print(f"{'IC Sharpe (neutral)':30s} {results_old2['ic_neutral'].mean()/results_old2['ic_neutral'].std():>10.3f}  {results_new2['ic_neutral'].mean()/results_new2['ic_neutral'].std():>10.3f}")

In [ ]:
XGB_PARAMS_FIXED = {
    'n_estimators': 300,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'min_child_weight': 50,
    'objective': 'binary:logistic',
    'device': 'cuda',
    'random_state': 42,
    'verbosity': 0,
}

def run_cv_fixed(df, regime_name):
    results = []

    for test_year in TEST_YEARS:
        train = df[df['year'] < test_year].copy()
        test  = df[df['year'] == test_year].copy()

        if len(train) == 0 or len(test) == 0:
            print(f"  {test_year}: SKIPPED — empty split")
            continue

        X_train    = train[FEATURES].values
        y_train    = train['label_5d_up'].values
        X_test     = test[FEATURES].values
        y_test_raw = test['ret_5d_f'].values
        y_test_neu = test['ret_5d_sector_neutral'].values

        model = xgb.XGBClassifier(**XGB_PARAMS_FIXED)
        model.fit(X_train, y_train, verbose=False)

        y_pred        = model.predict_proba(X_test)[:, 1]
        ic_raw, _     = spearmanr(y_test_raw, y_pred)
        ic_neutral, _ = spearmanr(y_test_neu, y_pred)

        results.append({
            'test_year':  test_year,
            'n_trees':    300,
            'ic_raw':     ic_raw,
            'ic_neutral': ic_neutral,
        })

        print(f"  {test_year}: trees= 300  IC_raw={ic_raw:+.4f}  IC_neutral={ic_neutral:+.4f}")

    return pd.DataFrame(results)

print("=" * 65)
print("FIXED: n_estimators=300, no early stopping, full train set")
print("=" * 65)
results_fixed = run_cv_fixed(df, 'fixed_300')

print("\n=== SUMMARY — Fixed 300 trees vs Old/New regimes ===")
print(f"{'':30s} {'OLD':>10s}  {'NEW':>10s}  {'FIXED':>10s}")
print(f"{'Mean IC_raw':30s} {results_old2['ic_raw'].mean():>10.4f}  {results_new2['ic_raw'].mean():>10.4f}  {results_fixed['ic_raw'].mean():>10.4f}")
print(f"{'Std IC_raw':30s} {results_old2['ic_raw'].std():>10.4f}  {results_new2['ic_raw'].std():>10.4f}  {results_fixed['ic_raw'].std():>10.4f}")
print(f"{'IC Sharpe (raw)':30s} {results_old2['ic_raw'].mean()/results_old2['ic_raw'].std():>10.3f}  {results_new2['ic_raw'].mean()/results_new2['ic_raw'].std():>10.3f}  {results_fixed['ic_raw'].mean()/results_fixed['ic_raw'].std():>10.3f}")
print(f"{'Mean IC_neutral':30s} {results_old2['ic_neutral'].mean():>10.4f}  {results_new2['ic_neutral'].mean():>10.4f}  {results_fixed['ic_neutral'].mean():>10.4f}")
print(f"{'Std IC_neutral':30s} {results_old2['ic_neutral'].std():>10.4f}  {results_new2['ic_neutral'].std():>10.4f}  {results_fixed['ic_neutral'].std():>10.4f}")
print(f"{'IC Sharpe (neutral)':30s} {results_old2['ic_neutral'].mean()/results_old2['ic_neutral'].std():>10.3f}  {results_new2['ic_neutral'].mean()/results_new2['ic_neutral'].std():>10.3f}  {results_fixed['ic_neutral'].mean()/results_fixed['ic_neutral'].std():>10.3f}")

print("\n=== Per-fold IC_raw comparison ===")
compare = results_old2[['test_year','ic_raw']].copy()
compare.columns = ['test_year', 'ic_raw_old']
compare['ic_raw_new']   = results_new2['ic_raw'].values
compare['ic_raw_fixed'] = results_fixed['ic_raw'].values
print(compare.to_string(index=False))

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("validation_regime_comparison")

# ── Regime metadata ──────────────────────────────────────────
regime_meta = {
    'old': {
        'results':     results_old2,
        'description': 'Last 10pct of train as val — early stopping',
        'run_name':    'old_last10pct_early_stopping',
    },
    'new': {
        'results':     results_new2,
        'description': 'Dedicated val year — early stopping',
        'run_name':    'new_dedicated_val_year_early_stopping',
    },
    'fixed': {
        'results':     results_fixed,
        'description': 'Fixed 300 trees — no early stopping — full train',
        'run_name':    'fixed_300_no_early_stopping',
    },
}

for key, meta in regime_meta.items():
    r   = meta['results']
    ic  = r['ic_raw']
    icn = r['ic_neutral']

    with mlflow.start_run(run_name=meta['run_name']):

        # ── Parameters ───────────────────────────────────────
        mlflow.log_param('regime',          key)
        mlflow.log_param('description',     meta['description'])
        mlflow.log_param('model',           'XGBClassifier')
        mlflow.log_param('n_estimators',    300 if key == 'fixed' else 'early_stopping')
        mlflow.log_param('early_stopping',  False if key == 'fixed' else True)
        mlflow.log_param('max_depth',       4)
        mlflow.log_param('learning_rate',   0.05)
        mlflow.log_param('subsample',       0.7)
        mlflow.log_param('colsample_bytree',0.7)
        mlflow.log_param('reg_alpha',       0.1)
        mlflow.log_param('reg_lambda',      1.0)
        mlflow.log_param('min_child_weight',50)
        mlflow.log_param('target_train',    'label_5d_up')
        mlflow.log_param('target_eval_raw', 'ret_5d_f')
        mlflow.log_param('target_eval_neu', 'ret_5d_sector_neutral')
        mlflow.log_param('test_years',      '2019-2025')
        mlflow.log_param('n_folds',         7)
        mlflow.log_param('features',        23)
        mlflow.log_param('universe',        '2335 tickers')
        mlflow.log_param('db',              'kairos_research.duckdb')

        # ── Summary metrics ───────────────────────────────────
        mlflow.log_metric('mean_ic_raw',         ic.mean())
        mlflow.log_metric('std_ic_raw',          ic.std())
        mlflow.log_metric('ic_sharpe_raw',       ic.mean() / ic.std())
        mlflow.log_metric('min_ic_raw',          ic.min())
        mlflow.log_metric('max_ic_raw',          ic.max())
        mlflow.log_metric('pct_positive_ic_raw', (ic > 0).mean())

        mlflow.log_metric('mean_ic_neutral',         icn.mean())
        mlflow.log_metric('std_ic_neutral',          icn.std())
        mlflow.log_metric('ic_sharpe_neutral',       icn.mean() / icn.std())
        mlflow.log_metric('min_ic_neutral',          icn.min())
        mlflow.log_metric('max_ic_neutral',          icn.max())
        mlflow.log_metric('pct_positive_ic_neutral', (icn > 0).mean())

        # ── Per-fold metrics ──────────────────────────────────
        for _, row in r.iterrows():
            yr = int(row['test_year'])
            mlflow.log_metric(f'ic_raw_{yr}',     row['ic_raw'])
            mlflow.log_metric(f'ic_neutral_{yr}', row['ic_neutral'])
            if 'n_trees' in row:
                mlflow.log_metric(f'n_trees_{yr}', row['n_trees'])

        # ── Per-fold detail as artifact ───────────────────────
        fold_path = f'/tmp/fold_detail_{key}.csv'
        r.to_csv(fold_path, index=False)
        mlflow.log_artifact(fold_path)

        print(f"Logged: {meta['run_name']}")
        print(f"  IC_raw Sharpe:     {ic.mean()/ic.std():.3f}")
        print(f"  IC_neutral Sharpe: {icn.mean()/icn.std():.3f}")
        print()

print("=== All three regimes logged to MLflow ===")
print("View at: http://localhost:5000")
print("Experiment: validation_regime_comparison")